**Installing dependencies**


pypdf: responsible for splitting and transforming PDF files

docx2txt: extracts text from your docx file

google-generativeai: gives access to the model

langchain: acts as a wrapper to the LLM (connects everything together)

chromadb: open-source vector database for storing and querying embeddings

langchain-community: loads data into the standard LangChain document format

langchain-google-genai: packages connecting Gemini and LangChain (lets us use Gemini for both the chat model and embeddings)

"langchain-chroma>=0.1.2": to access the Chroma vector store

python-dotenv: lets you store your Gemini API key in a separate .env file (no need in collab)

In [17]:
pip install pypdf docx2txt google-generativeai langchain chromadb langchain-community langchain-google-genai "langchain-chroma>=0.1.2" python-dotenv

---- Loading Gemini API KEY ----

In [2]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

---- Load Document ----


In [9]:
# Upload the document
from google.colab import files
uploaded= files.upload()

#Print and store the the uploaded filename
filename = list(uploaded.keys())[0]
filepath=f'/content/{filename}'

print(f' Uploaded:{filename}')

Saving the_lighthouse_keeper.pdf to the_lighthouse_keeper.pdf
 Uploaded:the_lighthouse_keeper.pdf


In [10]:
#Defining Document loader - LangChain's document format
import os

def load_document(file):
  name, extension=os.path.splitext(file)

  if extension == '.pdf':
    from langchain_community.document_loaders import PyPDFLoader
    print(f'Loading {file}')
    loader=PyPDFLoader(file)

  elif extension == '.docx':
    from langchain_community.document_loaders import Docx2txtLoader
    print(f'Loading {file}')
    loader = Docx2txtLoader(file)

  elif extension == '.txt':
    from langchain_community.document_loaders import TextLoader
    loader = TextLoader(file)

  else:
    print('Document type is not supported')
    return None

  data = loader.load()
  return data


In [12]:
#Load the uploaded file
data=load_document(filepath)
print(f' Number of pages: {len(data)}')
print(f' Page 0: {data[0].page_content[:500]}')


Loading /content/the_lighthouse_keeper.pdf
 Number of pages: 5
 Page 0: The Lighthouse Keeper of Ardmore
 Point
A short story
Chapter 1: The Storm That Never Left
The sea had been angry for three days. Not the kind of anger that arrives quickly and leaves
just as fast, but the slow, grinding fury of something that had decided to stay. Maren stood at
the top of the Ardmore Point lighthouse, watching the waves throw themselves at the rocks
below like they had a grudge to settle.
She had been the keeper of this lighthouse for eleven years, ever since her father had han


---- Create Data Chunks----

Full pages are too large to pass into an embedding model efficiently. The model works better with smaller, focused pieces of text. So we split each page into smaller overlapping chunks. The overlap (15% of chunk size) ensures that if a sentence gets cut at a chunk boundary, the next chunk still captures enough context to make sense of it.

In [19]:
def chunk_data (data, chunk_size=256) : #256 ensures easy retrieval. Trade-off: may cut many sentences midway
  from langchain_text_splitters import RecursiveCharacterTextSplitter

  sample_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap= int (chunk_size * 0.15))

  chunks = sample_splitter.split_documents(data)
  return chunks

In [20]:
chunks = chunk_data(data, chunk_size=256)

print (f' Total chunks created : {len(chunks)}')
print (f' Sample chuunk: \n {chunks[0].page_content}')

 Total chunks created : 30
 Sample chuunk: 
 The Lighthouse Keeper of Ardmore
 Point
A short story
Chapter 1: The Storm That Never Left
The sea had been angry for three days. Not the kind of anger that arrives quickly and leaves


---- **Create embeddings and store in ChromaDB** ----

We need to convert our text chunks into numbers (vectors/embeddings) that capture the meaning of the text. ChromaDB then stores these vectors locally on disk. Later when asked a question, the question also gets converted into a vector and ChromaDB finds the chunks whose vectors are closest to it, that's how it retrieves relevant context.

In [32]:
def create_embeddings_chroma(chunks, persist_directory='./chroma_db'):
  from langchain_chroma import Chroma
  from langchain_google_genai import GoogleGenerativeAIEmbeddings

  #Convert text chunks into vectors using Gemini Embeddings
  embeddings = GoogleGenerativeAIEmbeddings(
      model = 'models/gemini-embedding-001',
      google_api_key = os.environ["GEMINI_API_KEY"]
  )

  #Store the vectors in ChromaDB locally
  chroma_index = Chroma.from_documents(chunks, embeddings, persist_directory=persist_directory)

  return chroma_index

In [33]:
vector_store = create_embeddings_chroma(chunks)
print('Embeddings created and stored in ChromaDB successfully.')

Embeddings created and stored in ChromaDB successfully.


---- Query and get Answer ----

In [42]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def ask_and_get_answer(vector_store, q, k=3):


    llm = ChatGoogleGenerativeAI(
        model='gemini-2.0-flash',
        google_api_key=os.environ["GEMINI_API_KEY"],
        temperature=0.0
    )

    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    prompt = ChatPromptTemplate.from_template("""
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    Question: {input}""")

    # Modern LCEL chain - no langchain.chains needed
    chain = (
        {"context": retriever, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    response = chain.invoke(q)
    return response

In [43]:
#Testing
q = 'Who is Maren and what does she do?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

Maren is at the top of the Ardmore Point lighthouse, watching the waves. She also watched the white tower catching the morning sun. She knows that the light rotates every four seconds.


---- Adding Chat history memory ----

Right now if you ask a follow-up question like "what did she do next?" the model has no idea what "she" refers to. Every question is treated independently. We need to add memory so the model can reference previous exchanges in the conversation. We do this by maintaining a chat_history list that grows with every question and answer.

In [46]:
from langchain_core.prompts import  MessagesPlaceholder
from langchain_core.runnables import  RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.0
)

retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})

prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the provided context.
    If you don't know the answer, just say you don't know.\n\n{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = (
    {
        # Extract only the question string for the retriever
        "context": RunnableLambda(lambda x: x["input"]) | retriever,
        "input": RunnableLambda(lambda x: x["input"]),
        "chat_history": RunnableLambda(lambda x: x["chat_history"])
    }
    | prompt
    | llm
    | StrOutputParser()
)

chat_history = []

def ask_with_memory(question):
    response = chain.invoke({
        "input": question,
        "chat_history": chat_history
    })
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    return response

In [47]:
#Testing with follow-up question

answer1= ask_with_memory('Who is Maren?')
print(answer1)
print ('-' * 50)

answer2 = ask_with_memory('What happened to her father?')
print(answer2)
print('-' * 50)

answer3 = ask_with_memory('What did she do after that?')
print(answer3)


Maren is at the top of the Ardmore Point lighthouse. She is watching the waves. She is not a warm person by nature, but she is a practical one. She made tea for a man who had nearly drowned. She watched the white tower catching the morning sun. She knew the light rotated every four seconds the way she knew her own heartbeat.
--------------------------------------------------
Her father handed her the iron key ring and said, without ceremony, 'It's yours now.' He had walked down the spiral staircase, climbed into his boat, and sailed north. She had never heard from him again.
--------------------------------------------------
She didn't answer, but she thought about it for the rest of the week. She thought about what he said at the top of the lighthouse almost every day since. She folded the letter, put it in an envelope, and walked down the cliff path to the postbox in the village. On the way back, she stopped at the top of the path and looked out at the sea.


---- Interactive Question Loop ----

In [48]:
print("Ask anything about the document. Type 'exit' to quit.\n")

while True:
    query = input('Your question: ')

    if query.lower() in ['exit', 'quit', 'bye']:
        print('\nBye bye!')
        break

    answer = ask_with_memory(query)
    print(f'\nAnswer: {answer}')
    print('-' * 50)

Ask anything about the document. Type 'exit' to quit.

Your question: what is this story?

Answer: This story is "The Lighthouse Keeper of Ardmore Point."
--------------------------------------------------
Your question: How many pages total?

Answer: The total number of pages is 5.
--------------------------------------------------
Your question: what the theme?

Answer: Based on the context, a possible theme is finding one's way home or the responsibility of keeping the light, both literally and figuratively. The text also hints at themes of abandonment, duty, and the passage of time.
--------------------------------------------------
Your question: is the story good?

Answer: I am unable to determine if the story is good or not.
--------------------------------------------------
Your question: Thank you

Answer: You're welcome.
--------------------------------------------------
Your question: Bye
Bye bye!
